In [14]:
from pathlib import Path
import pandas as pd
import folium
from folium import FeatureGroup
from folium.plugins import PolyLineTextPath
from IPython.display import display, IFrame


# =========================================================
# helper
# =========================================================

def add_solution_to_map(
    m,
    result_dir,
    solution_name,
    color_palette
):

    result_dir = Path(result_dir)

    deliveries = pd.read_csv(result_dir / "deliveries.csv")
    routes = pd.read_csv(result_dir / "routes.csv")
    cafes = pd.read_csv(result_dir / "served_cafes.csv")

    summary = (
        deliveries
        .groupby(["van_id", "cafe_name"], as_index=False)
        .agg(
            total_weight_kg=("total_weight_kg", "sum"),
            total_margin=("total_margin", "sum"),
            n_products=("product_id", "nunique")
        )
    )

    cafes_plot = cafes.merge(
        summary,
        left_on=["van_id", "name"],
        right_on=["van_id", "cafe_name"],
        how="left"
    )

    max_margin = cafes_plot["total_margin"].fillna(0).max()

    solution_group = FeatureGroup(
        name=f"{solution_name}",
        show=True
    )

    # =====================================================
    # depot
    # =====================================================

    depot_rows = cafes_plot[cafes_plot["node_type"] == "depot"]

    for _, row in depot_rows.iterrows():

        folium.Marker(
            location=[row["latitude"], row["longitude"]],
            popup=f"{solution_name} Depot",
            tooltip=f"{solution_name} Depot",
            icon=folium.Icon(
                color="black",
                icon="home",
                prefix="fa"
            )
        ).add_to(solution_group)

    # =====================================================
    # routes
    # =====================================================

    for van_id in sorted(routes["van_id"].dropna().unique()):

        color = color_palette.get(van_id, "gray")

        van_routes = routes[routes["van_id"] == van_id]

        for _, r in van_routes.iterrows():

            locations = [
                [r["from_latitude"], r["from_longitude"]],
                [r["to_latitude"], r["to_longitude"]],
            ]

            line = folium.PolyLine(
                locations=locations,
                color=color,
                weight=4,
                opacity=0.75,
                tooltip=(
                    f"{solution_name}<br>"
                    f"{van_id}<br>"
                    f"{r['from_node']} → {r['to_node']}<br>"
                    f"{r['distance_km']:.2f} km"
                )
            )

            line.add_to(solution_group)

            PolyLineTextPath(
                line,
                " → ",
                repeat=True,
                offset=7,
                attributes={
                    "fill": color,
                    "font-size": "12",
                    "font-weight": "bold"
                }
            ).add_to(solution_group)

    # =====================================================
    # cafes
    # =====================================================

    cafe_rows = cafes_plot[cafes_plot["node_type"] == "cafe"]

    for _, row in cafe_rows.iterrows():

        margin = (
            0 if pd.isna(row["total_margin"])
            else row["total_margin"]
        )

        radius = 5 + 10 * (
            margin / max_margin
            if max_margin > 0 else 0
        )

        cafe_delivery = deliveries[
            (deliveries["van_id"] == row["van_id"]) &
            (deliveries["cafe_name"] == row["name"])
        ]

        product_table = cafe_delivery[
            ["product_id", "quantity", "total_margin"]
        ].to_html(
            index=False,
            float_format=lambda x: f"{x:.2f}",
            border=0
        )

        popup = f"""
        <b>{solution_name}</b><br>
        <b>{row['name']}</b><br>
        Van: {row['van_id']}<br>
        Weight: {row['total_weight_kg']:.2f} kg<br>
        Margin: ${row['total_margin']:.2f}<br>
        Products: {row['n_products']}<br><br>
        {product_table}
        """

        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=radius,
            color=color_palette.get(row["van_id"], "gray"),
            fill=True,
            fill_opacity=0.85,
            popup=folium.Popup(popup, max_width=450),
            tooltip=(
                f"{solution_name} | "
                f"{row['name']} | "
                f"${margin:.2f}"
            )
        ).add_to(solution_group)

    solution_group.add_to(m)


# =========================================================
# create map
# =========================================================

m = folium.Map(
    location=[-37.80, 144.97],
    zoom_start=13,
    tiles="CartoDB positron"
)

# =========================================================
# color palettes
# =========================================================

# MTZ = cool colors
mtz_colors = {
    "van_1": "blue",
    "van_2": "darkblue",
    "van_3": "cadetblue",
    "van_4": "purple",
    "van_5": "green",
    "van_6": "darkgreen",
}

# DFJ = warm colors
dfj_colors = {
    "van_1": "red",
    "van_2": "darkred",
    "van_3": "orange",
    "van_4": "lightred",
    "van_5": "pink",
    "van_6": "beige",
}

# =========================================================
# add MTZ
# =========================================================

add_solution_to_map(
    m,
    "../results/mtz",
    "MTZ",
    mtz_colors
)

# =========================================================
# add DFJ
# =========================================================

add_solution_to_map(
    m,
    "../results/dfj",
    "DFJ",
    dfj_colors
)

# =========================================================
# save
# =========================================================

OUT_PATH = Path("../results/maps/comparison_map.html")

m.save(OUT_PATH)

print("Saved to:")
print(OUT_PATH)


Saved to:
..\results\maps\comparison_map.html


In [15]:
from pathlib import Path
import pandas as pd
import folium
from folium.plugins import PolyLineTextPath
from IPython.display import display, IFrame


def build_single_route_map(result_dir, output_path, solution_name, color_palette):
    result_dir = Path(result_dir)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    deliveries = pd.read_csv(result_dir / "deliveries.csv")
    routes = pd.read_csv(result_dir / "routes.csv")
    cafes = pd.read_csv(result_dir / "served_cafes.csv")

    summary = (
        deliveries
        .groupby(["van_id", "cafe_name"], as_index=False)
        .agg(
            total_weight_kg=("total_weight_kg", "sum"),
            total_margin=("total_margin", "sum"),
            n_products=("product_id", "nunique")
        )
    )

    cafes_plot = cafes.merge(
        summary,
        left_on=["van_id", "name"],
        right_on=["van_id", "cafe_name"],
        how="left"
    )

    m = folium.Map(
        tiles="CartoDB positron"
    )

    m.fit_bounds([
        [cafes_plot["latitude"].min(), cafes_plot["longitude"].min()],
        [cafes_plot["latitude"].max(), cafes_plot["longitude"].max()]
    ])

    max_margin = cafes_plot["total_margin"].fillna(0).max()

    # -------------------------
    # depot
    # -------------------------
    depot_rows = cafes_plot[cafes_plot["node_type"] == "depot"]

    for _, row in depot_rows.iterrows():
        folium.Marker(
            location=[row["latitude"], row["longitude"]],
            popup=f"<b>{row['name']}</b><br>{solution_name} Depot",
            tooltip=f"{solution_name} Depot",
            icon=folium.Icon(
                color="red",
                icon="star",
                prefix="fa"
            )
        ).add_to(m)

    # -------------------------
    # routes
    # -------------------------
    for van_id in sorted(routes["van_id"].dropna().unique()):
        color = color_palette.get(van_id, "gray")
        van_routes = routes[routes["van_id"] == van_id]

        for _, r in van_routes.iterrows():
            locations = [
                [r["from_latitude"], r["from_longitude"]],
                [r["to_latitude"], r["to_longitude"]],
            ]

            line = folium.PolyLine(
                locations=locations,
                color=color,
                weight=4,
                opacity=0.75,
                tooltip=(
                    f"{solution_name}<br>"
                    f"{van_id}<br>"
                    f"{r['from_node']} → {r['to_node']}<br>"
                    f"Distance: {r['distance_km']:.2f} km<br>"
                    f"Travel time: {r['travel_time_hours']:.2f} h"
                )
            )

            line.add_to(m)

            PolyLineTextPath(
                line,
                " → ",
                repeat=True,
                offset=7,
                attributes={
                    "fill": color,
                    "font-size": "12",
                    "font-weight": "bold"
                }
            ).add_to(m)

    # -------------------------
    # cafes
    # -------------------------
    cafe_rows = cafes_plot[cafes_plot["node_type"] == "cafe"]

    for _, row in cafe_rows.iterrows():
        margin = 0 if pd.isna(row["total_margin"]) else row["total_margin"]

        radius = 5 + 10 * (
            margin / max_margin
            if max_margin > 0 else 0
        )

        cafe_delivery = deliveries[
            (deliveries["van_id"] == row["van_id"]) &
            (deliveries["cafe_name"] == row["name"])
        ]

        product_table = cafe_delivery[
            ["product_id", "quantity", "total_weight_kg", "total_margin"]
        ].to_html(
            index=False,
            float_format=lambda x: f"{x:.2f}",
            border=0
        )

        popup = f"""
        <b>{row['name']}</b><br>
        Solution: {solution_name}<br>
        Van: {row['van_id']}<br>
        Weight: {row['total_weight_kg']:.2f} kg<br>
        Margin: ${row['total_margin']:.2f}<br>
        Products: {row['n_products']}<br><br>
        <b>Product details</b>
        {product_table}
        """

        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=radius,
            color=color_palette.get(row["van_id"], "gray"),
            fill=True,
            fill_opacity=0.85,
            popup=folium.Popup(popup, max_width=550),
            tooltip=f"{row['name']} | {row['van_id']} | ${margin:.2f}"
        ).add_to(m)

    # -------------------------
    # layer control + save
    # -------------------------
    folium.LayerControl(collapsed=False).add_to(m)

    m.save(output_path)

    print(f"{solution_name} map saved to: {output_path}")

    return m


mtz_colors = {
    "van_1": "blue",
    "van_2": "darkblue",
    "van_3": "cadetblue",
    "van_4": "purple",
    "van_5": "green",
    "van_6": "darkgreen",
}

dfj_colors = {
    "van_1": "red",
    "van_2": "darkred",
    "van_3": "orange",
    "van_4": "lightred",
    "van_5": "pink",
    "van_6": "beige",
}


mtz_map = build_single_route_map(
    result_dir="../results/mtz",
    output_path="../results/maps/mtz_route_map.html",
    solution_name="MTZ",
    color_palette=mtz_colors
)

dfj_map = build_single_route_map(
    result_dir="../results/dfj",
    output_path="../results/maps/dfj_route_map.html",
    solution_name="DFJ",
    color_palette=dfj_colors
)

MTZ map saved to: ..\results\maps\mtz_route_map.html
DFJ map saved to: ..\results\maps\dfj_route_map.html


In [16]:
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

# =========================================================
# Paths
# =========================================================

MAP_DIR = Path("../results/maps")

HTML_FILES = [
    MAP_DIR / "comparison_map.html",
    MAP_DIR / "mtz_route_map.html",
    MAP_DIR / "dfj_route_map.html",
]

# =========================================================
# Chrome options
# =========================================================

options = Options()

options.add_argument("--headless")

# good report resolution
options.add_argument("--window-size=1400,1000")

# retina / HiDPI rendering
options.add_argument("--force-device-scale-factor=2")

driver = webdriver.Chrome(options=options)

driver.set_window_size(1400, 1000)

# =========================================================
# Export each html to png
# =========================================================

for html_path in HTML_FILES:

    png_path = html_path.with_suffix(".png")

    # open html
    driver.get(f"file://{html_path.resolve()}")

    # wait for map tiles to load
    time.sleep(4)

    # save screenshot
    driver.save_screenshot(str(png_path))

    print(f"Saved PNG to: {png_path}")

# =========================================================
# Close browser
# =========================================================

driver.quit()

print("All PNG exports completed.")

Saved PNG to: ..\results\maps\comparison_map.png
Saved PNG to: ..\results\maps\mtz_route_map.png
Saved PNG to: ..\results\maps\dfj_route_map.png
All PNG exports completed.
